code fonctionne pour scrapper liens moi de oct

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# LISTE DES URLs
# --------------------------

urls = [
    "https://x.com/hespress/status/1976210649551192205",
    "https://x.com/hespress/status/1979130334152192034",
    "https://x.com/hespress/status/1978574144884113491",
    "https://x.com/hespress/status/1976297129363571082",
    "https://x.com/hespress/status/1975943014091571370",
    "https://x.com/hespress/status/1977698777730277566",
    "https://x.com/hespress/status/1979306454076023180",
    "https://x.com/hespress/status/1979336639190016254",
    "https://x.com/hespress/status/1978151334315458720",
    "https://x.com/hespress/status/1976633938966716531",
    "https://x.com/hespress/status/1978073520014840140",
    "https://x.com/hespress/status/1976266395261284555",
    "https://x.com/hespress/status/1978913891280146561",
    "https://x.com/hespress/status/1977766109479617012",
    "https://x.com/hespress/status/1978415364955373758",
    "https://x.com/hespress/status/1976978391976100285",
    "https://x.com/hespress/status/1978812937050456163",
    "https://x.com/hespress/status/1976260102853173348",
    "https://x.com/hespress/status/1978513671396483165",
    "https://x.com/hespress/status/1976710748949618942",
    "https://x.com/hespress/status/1976585078340739449",
    "https://x.com/hespress/status/1976033588455112795",
    "https://x.com/hespress/status/1976948239242084740",
    "https://x.com/hespress/status/1977819152849993859",
    "https://x.com/hespress/status/1978506226620105189",
    "https://x.com/hespress/status/1978032466976805046",
    "https://x.com/hespress/status/1975909050660356491",
    "https://x.com/hespress/status/1976754628596289810",
    "https://x.com/hespress/status/1976686211528872142",
    "https://x.com/hespress/status/1976305628856582534",
    "https://x.com/hespress/status/1979230988912808124",
    "https://x.com/hespress/status/1978079930459066452",
    "https://x.com/hespress/status/1976209234132922522",
    "https://x.com/hespress/status/1977372092329820406",
    "https://x.com/hespress/status/1978474786834555072",
    "https://x.com/hespress/status/1976596107917422696",
    "https://x.com/hespress/status/1975901492885487743",
    "https://x.com/hespress/status/1978551521190003090",
    "https://x.com/hespress/status/1977678930527330399",
    "https://x.com/hespress/status/1978050596281208835",
    "https://x.com/hespress/status/1976662037884186859",
    "https://x.com/hespress/status/1975916589661778301",
    "https://x.com/hespress/status/1976040050371793072",
    "https://x.com/hespress/status/1977702496798732729",
    "https://x.com/hespress/status/1976771524314153397",
    "https://x.com/hespress/status/1978442765106037123",
    "https://x.com/hespress/status/1975969443454079483",
    "https://x.com/hespress/status/1978604313912959119",
    "https://x.com/hespress/status/1978063301503250466",
    "https://x.com/hespress/status/1978046020865986641",
    "https://x.com/hespress/status/1976611174331338850",
    "https://x.com/hespress/status/1976245003392516379",
    "https://x.com/hespress/status/1977705226279825904",
    "https://x.com/hespress/status/1977688034633216343",
    "https://x.com/hespress/status/1979197164757606428",
    "https://x.com/hespress/status/1979276236707738103",
    "https://x.com/hespress/status/1978804424513401170",
    "https://x.com/hespress/status/1978755959720722651",
    "https://x.com/hespress/status/1978521334591193592",
    "https://x.com/hespress/status/1977719712608821550",
    "https://x.com/hespress/status/1978037619637240276",
    "https://x.com/hespress/status/1978123020045574543"
]
# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver_with_profile():
    """Configure le driver avec un chemin de profil valide"""
    options = Options()
    
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    os.makedirs(user_profile_dir, exist_ok=True)
    
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 1,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=fr")
        return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    """Attend que l'utilisateur se connecte manuellement à Twitter"""
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print("⏳ Le script attendra 2 minutes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            elif "twitter.com" in current_url or "x.com" in current_url:
                print("✅ Déjà connecté ou page d'accueil chargée")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠️ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# FONCTIONS UTILITAIRES (conservées identiques)
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s:
        return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

# --------------------------
# FONCTIONS D'EXTRACTION (conservées identiques)
# --------------------------
def extract_main_tweet_data(driver):
    """Extrait spécifiquement le PREMIER tweet (tweet principal)"""
    try:
        wait = WebDriverWait(driver, 20)
        time.sleep(3)
        
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        
        if not all_tweets:
            logging.error("❌ Aucun tweet trouvé sur la page")
            return None
        
        main_tweet_element = all_tweets[0]
        logging.info(f"✅ {len(all_tweets)} tweets trouvés, extraction du premier (tweet principal)")
        
        tweet_data = {}
        
        # Auteur du tweet
        try:
            author_element = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            full_text = author_element.text
            lines = full_text.split('\n')
            tweet_data['auteur'] = lines[0] if lines else "Non trouvé"
            
            author_links = main_tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href and href.count('/') >= 3:
                    username = href.rstrip('/').split('/')[-1]
                    if username and not username.startswith('status'):
                        tweet_data['compte'] = username
                        break
            
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
                
            logging.info(f"📝 Auteur trouvé: {tweet_data['auteur']} (@{tweet_data['compte']})")
        except Exception as e:
            logging.error(f"Erreur extraction auteur: {e}")
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet principal
        try:
            content = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
            logging.info(f"📄 Contenu trouvé: {tweet_data['contenu'][:100]}...")
        except Exception as e:
            logging.warning(f"Contenu non trouvé: {e}")
            tweet_data['contenu'] = ""
        
        # Date et heure
        try:
            time_element = main_tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
            logging.info(f"🕒 Date trouvée: {tweet_data['heure_affichage']}")
        except Exception as e:
            logging.warning(f"Date non trouvée: {e}")
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"❌ Erreur lors de l'extraction du tweet principal: {e}")
        return None

def get_video_duration_js(driver):
    """Récupère la durée de la vidéo principale"""
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        if (vids.length > 0) {
            let mainVideo = vids[0];
            return mainVideo.duration || 0;
        }
        return 0;
        """
        duration = driver.execute_script(script)
        if duration and duration > 0 and duration < 60*60*10:
            logging.info(f"🎬 Durée vidéo trouvée: {int(duration)}s")
            return int(round(duration))
    except Exception as e:
        logging.warning(f"Erreur get_video_duration_js: {e}")
    return 0

def get_views(driver):
    """Récupère les vues en cherchant dans plusieurs emplacements"""
    try:
        view_spans = driver.find_elements(By.XPATH, "//span[contains(text(), 'Views') or contains(text(), 'views') or contains(text(), 'Vues') or contains(text(), 'vues')]")
        for span in view_spans:
            parent = span.find_element(By.XPATH, "./..")
            aria_label = parent.get_attribute("aria-label") or ""
            num = parse_number_from_text(aria_label)
            if num > 0:
                logging.info(f"👁️ Vues trouvées (méthode 1): {num}")
                return num
    except:
        pass

    try:
        all_elements = driver.find_elements(By.XPATH, "//*[@aria-label]")
        for el in all_elements:
            aria = el.get_attribute("aria-label") or ""
            if any(keyword in aria.lower() for keyword in ['views', 'vues', 'vue']):
                num = parse_number_from_text(aria)
                if num > 0:
                    logging.info(f"👁️ Vues trouvées (méthode 2): {num}")
                    return num
    except:
        pass

    logging.warning("⚠️ Nombre de vues non trouvé")
    return 0

def extract_main_tweet_metrics(driver):
    """Extrait les métriques du PREMIER tweet (tweet principal)"""
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Views": 0}
    try:
        time.sleep(2)
        
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        if not all_tweets:
            logging.error("❌ Aucun tweet pour extraire les métriques")
            return metrics
        
        main_tweet = all_tweets[0]
        logging.info("📊 Extraction des métriques du premier tweet...")
        
        # Replies
        try:
            reply_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="reply"]')
            aria_label = reply_button.get_attribute('aria-label') or ""
            metrics["Replies"] = parse_number_from_text(aria_label)
            logging.info(f"💬 Replies: {metrics['Replies']}")
        except Exception as e:
            logging.debug(f"Replies non trouvés: {e}")
        
        # Retweets
        try:
            retweet_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="retweet"]')
            aria_label = retweet_button.get_attribute('aria-label') or ""
            metrics["Reposts"] = parse_number_from_text(aria_label)
            logging.info(f"🔄 Retweets: {metrics['Reposts']}")
        except Exception as e:
            logging.debug(f"Retweets non trouvés: {e}")
        
        # Likes
        try:
            like_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="like"]')
            aria_label = like_button.get_attribute('aria-label') or ""
            metrics["Likes"] = parse_number_from_text(aria_label)
            logging.info(f"❤️ Likes: {metrics['Likes']}")
        except Exception as e:
            logging.debug(f"Likes non trouvés: {e}")
        
        # Views
        metrics["Views"] = get_views(driver)
        
    except Exception as e:
        logging.error(f"❌ Erreur extraction métriques: {e}")
    
    return metrics

def get_comments_texts(driver, max_comments=200):
    """Récupère les commentaires (en EXCLUANT le premier tweet)"""
    comments_data = []
    try:
        logging.info("📜 Scroll pour charger les commentaires...")
        for i in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.5)

        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"📝 {len(tweet_elements)} tweets trouvés au total")
        
        if len(tweet_elements) <= 1:
            logging.warning("⚠️ Peu de commentaires trouvés")
            return comments_data
        
        for idx, tweet_element in enumerate(tweet_elements):
            if idx == 0:
                logging.info("⏭️ Premier tweet ignoré (tweet principal)")
                continue
                
            if len(comments_data) >= max_comments:
                break
                
            try:
                comment_data = extract_tweet_data(tweet_element)
                if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                    comment_data['langue'] = detect_langue(comment_data['contenu'])
                    comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                    comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                    comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire {idx}: {e}")
                continue
                
    except Exception as e:
        logging.error(f"❌ Erreur get_comments_texts: {e}")
    
    logging.info(f"✅ {len(comments_data)} commentaires récupérés")
    return comments_data

def extract_tweet_data(tweet_element):
    """Extrait les données d'un tweet (pour les commentaires)"""
    try:
        tweet_data = {}
        
        # Auteur
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            author_links = tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href:
                    username = href.rstrip('/').split('/')[-1]
                    if username:
                        tweet_data['compte'] = username
                        break
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
        except:
            tweet_data['date_publication'] = "Non trouvé"
        
        # Statistiques
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                aria_label = element.get_attribute('aria-label') or ""
                tweet_data['statistiques'][stat] = parse_number_from_text(aria_label)
            except:
                tweet_data['statistiques'][stat] = 0
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur extraction tweet: {e}")
        return None

def analyze_comments(comments_data):
    """Analyse les commentaires"""
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER MULTI-URLS AMÉLIORÉ
# --------------------------
def scraper_toutes_urls_automatique():
    """Scrape toutes les URLs automatiquement sans interruption"""
    
    print("=" * 60)
    print("🚀 SCRAPER TWITTER/X - MODE AUTOMATIQUE MULTI-URLS")
    print("=" * 60)
    print(f"📋 Nombre d'URLs à scraper: {len(urls)}")
    
    # Initialiser les listes pour stocker tous les résultats
    tous_les_resultats = []
    compteur_succes = 0
    compteur_echecs = 0
    
    # Configurer le driver une seule fois
    driver = setup_driver_with_profile()
    if driver is None:
        logging.error("❌ Impossible de créer le driver Chrome")
        return
    
    try:
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        logging.info("📥 Ouverture de Twitter/X...")
        driver.get("https://twitter.com")
        
        # Attendre la connexion manuelle une seule fois
        if not wait_for_manual_login(driver):
            logging.error("❌ Échec de la connexion")
            return
        
        # Scraper chaque URL
        for i, url in enumerate(urls, 1):
            print(f"\n{'='*60}")
            print(f"📊 Scraping {i}/{len(urls)}: {url}")
            print(f"{'='*60}")
            
            try:
                # Aller à l'URL spécifique
                driver.get(url)
                
                # Attendre le chargement complet
                time.sleep(8)
                logging.info("⏳ Attente du chargement de la page...")

                # Scroll léger pour s'assurer que tout est chargé
                for j in range(3):
                    driver.execute_script("window.scrollBy(0, 500);")
                    time.sleep(2)
                
                # Remonter en haut
                driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(2)

                data = {}
                
                # Extraire le tweet principal
                logging.info("🔍 Extraction du tweet principal...")
                main_tweet_data = extract_main_tweet_data(driver)
                
                if main_tweet_data:
                    data["Titre"] = main_tweet_data.get('contenu', '')
                    data["Auteur"] = main_tweet_data.get('auteur', '')
                    data["Compte"] = main_tweet_data.get('compte', '')
                    data["Date publication"] = main_tweet_data.get('date_publication', '')
                    logging.info(f"✅ Tweet principal extrait - Auteur: {data['Auteur']}")
                else:
                    logging.error("❌ Impossible d'extraire le tweet principal")
                    data["Titre"] = ""
                    data["Auteur"] = ""
                    data["Compte"] = ""
                    data["Date publication"] = ""

                # Catégorie
                data["Catégorie"] = detect_categorie(data.get("Titre", ""))

                # Durée vidéo
                dur = get_video_duration_js(driver)
                data["Durée"] = f"{dur}s" if dur else "Inconnue"

                # Métriques du tweet principal
                logging.info("📊 Extraction des métriques...")
                metrics = extract_main_tweet_metrics(driver)
                data["Likes"] = metrics.get("Likes", 0)
                data["Retweets"] = metrics.get("Reposts", 0)
                data["Commentaires"] = metrics.get("Replies", 0)
                data["Nombre de vues"] = metrics.get("Views", 0)
                data["Nombre de partages"] = metrics.get("Reposts", 0)
                data["Nombre de commentaires"] = metrics.get("Replies", 0)

                # Commentaires
                logging.info("💬 Extraction des commentaires...")
                comments_data = get_comments_texts(driver, max_comments=200)
                analyzed_comments, comment_stats = analyze_comments(comments_data)
                data["comments"] = analyzed_comments
                data["comment_stats"] = comment_stats

                # Mots les plus cités
                title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
                overall_counter = Counter(title_words)
                overall_counter.update(comment_stats.get("top_words", {}))
                data["Mots plus cités"] = dict(overall_counter.most_common(10))

                # Langue
                data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
                data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

                # Polarité
                data["Polarité"] = detect_polarite(data.get("Titre", ""))
                if data["Polarité"] > 0.1:
                    sent = "Positive"
                elif data["Polarité"] < -0.1:
                    sent = "Négative"
                else:
                    sent = "Neutre"
                data["% Polarité"] = {sent: 100}
                data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

                data["Lien tweet"] = url
                data["Statut"] = "succès"

                # Ajouter aux résultats
                tous_les_resultats.append(data)
                compteur_succes += 1
                
                logging.info(f"✅ URL {i}/{len(urls)} traitée avec succès")
                
                # Sauvegarde incrémentale après chaque URL
                with open("resultats_hespress_tous.json", "w", encoding="utf-8") as f:
                    json.dump(tous_les_resultats, f, ensure_ascii=False, indent=2)
                
                # Sauvegarde CSV incrémentale
                flat_data = []
                for result in tous_les_resultats:
                    flat = {
                        "Titre": result.get("Titre", ""),
                        "Auteur": result.get("Auteur", ""),
                        "Compte": result.get("Compte", ""),
                        "Catégorie": result.get("Catégorie", ""),
                        "Date publication": result.get("Date publication", ""),
                        "Durée": result.get("Durée", ""),
                        "Likes": result.get("Likes", 0),
                        "Retweets": result.get("Retweets", 0),
                        "Nombre de vues": result.get("Nombre de vues", 0),
                        "Nombre de partages": result.get("Nombre de partages", 0),
                        "Nombre de commentaires": result.get("Nombre de commentaires", 0),
                        "Mots plus cités": json.dumps(result.get("Mots plus cités", {}), ensure_ascii=False),
                        "Langue": result.get("Langue", ""),
                        "% Langues": json.dumps(result.get("% Langues", {}), ensure_ascii=False),
                        "Polarité": result.get("Polarité", 0),
                        "% Polarité": json.dumps(result.get("% Polarité", {}), ensure_ascii=False),
                        "Lien tweet": result.get("Lien tweet", ""),
                        "Statut": result.get("Statut", "succès")
                    }
                    flat_data.append(flat)

                df = pd.DataFrame(flat_data)
                df.to_csv("resultats_hespress_tous.csv", index=False, encoding="utf-8-sig")
                
                # Pause entre les requêtes pour éviter le blocage
                time.sleep(5)
                
            except Exception as e:
                logging.error(f"❌ Erreur sur l'URL {i}: {e}")
                tous_les_resultats.append({
                    "Lien tweet": url,
                    "Statut": "échec",
                    "Erreur": str(e)
                })
                compteur_echecs += 1
                
                # Sauvegarde même en cas d'erreur
                with open("resultats_hespress_tous.json", "w", encoding="utf-8") as f:
                    json.dump(tous_les_resultats, f, ensure_ascii=False, indent=2)
        
        # Résumé final
        print(f"\n{'='*60}")
        print("📊 RAPPORT FINAL")
        print(f"{'='*60}")
        print(f"✅ URLs réussies: {compteur_succes}")
        print(f"❌ URLs échouées: {compteur_echecs}")
        print(f"📈 Taux de réussite: {round(compteur_succes/len(urls)*100, 1)}%")
        print(f"💾 Fichiers sauvegardés:")
        print(f"   - resultats_hespress_tous.json")
        print(f"   - resultats_hespress_tous.csv")
        print(f"{'='*60}")
        
    except Exception as e:
        logging.error(f"❌ Erreur générale: {e}")
    finally:
        driver.quit()
        logging.info("🔒 Navigateur fermé")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    print("=" * 60)
    print("🚀 SCRAPER TWITTER/X - VERSION AMÉLIORÉE")
    print("=" * 60)
    print("Choisissez le mode:")
    print("1. Mode automatique multi-URLs (recommandé)")
    print("2. Mode manuel single URL")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        scraper_toutes_urls_automatique()
    elif choix == "2":
        url = input("Entrez l'URL à scraper: ").strip()
        if url:
            # Utiliser la fonction originale pour une seule URL
            driver = setup_driver_with_profile()
            if driver:
                # ... (code pour single URL)
                pass
        else:
            print("❌ URL invalide")
    else:
        print("❌ Choix invalide. Veuillez entrer 1 ou 2.")